[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/robomotic/mjlab/blob/feature/motor-database-extension/notebooks/electrical/02_intro.ipynb)

# Tutorial 2: Motor Sizing with CartPole

## 🎯 Goal

Learn why **proper motor sizing matters** by comparing three motors on the same task:

1. **Underpowered** (20 N⋅m peak) - struggles with the load
2. **Well-sized** (60 N⋅m peak) - handles load comfortably
3. **Overpowered** (150 N⋅m peak) - overkill for the task

We'll apply **sinusoidal torque commands** (cyclic loading to oscillate the cart) and compare:
- Torque tracking ability
- Current draw
- Power consumption
- Motor temperature

## 📚 What You'll Learn

- How to compare different motor configurations
- Why underpowered motors saturate and can't deliver full torque
- Why overpowered motors waste power
- The **2-4× sizing rule**: motors should operate at 25-50% of peak torque

This builds on **Tutorial 1** which covered motor physics fundamentals.

In [ ]:
# Check if running in Google Colab
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
  print("Running in Google Colab - installing mjlab...")
  !pip install -q git+https://github.com/robomotic/mjlab.git@feature/motor-database-extension

  # Download model files
  print("\nDownloading CartPole models with motor variants...")
  !mkdir -p notebooks/electrical/models
  !wget -q https://raw.githubusercontent.com/robomotic/mjlab/feature/motor-database-extension/notebooks/electrical/models/cartpole_underpowered.xml -O notebooks/electrical/models/cartpole_underpowered.xml
  !wget -q https://raw.githubusercontent.com/robomotic/mjlab/feature/motor-database-extension/notebooks/electrical/models/cartpole_good.xml -O notebooks/electrical/models/cartpole_good.xml
  !wget -q https://raw.githubusercontent.com/robomotic/mjlab/feature/motor-database-extension/notebooks/electrical/models/cartpole_overpowered.xml -O notebooks/electrical/models/cartpole_overpowered.xml

  print("✓ Installation complete!")
else:
  print("Running locally - assuming mjlab is already installed")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path

from mjlab.tasks.cartpole.cartpole_env_cfg import cartpole_constant_rotation_env_cfg
from mjlab.envs import ManagerBasedRlEnv

print("✓ All imports successful!")

## Step 1: Define Motor Configurations

We'll compare three motors with different peak torque ratings:

| Motor | Peak Torque | Kt (N⋅m/A) | Resistance (Ω) | Thermal Res (°C/W) |
|-------|-------------|------------|----------------|--------------------|
| **Underpowered** | 20 N⋅m | 1.2 | 1.0 | 8.0 |
| **Well-sized** | 60 N⋅m | 2.5 | 0.5 | 5.0 |
| **Overpowered** | 150 N⋅m | 3.5 | 0.3 | 3.0 |

All three will receive the **same torque command**: 15 N⋅m sinusoidal (oscillating the cart back and forth under cyclic loading).

The Kt values are derived from stall conditions (Kt = stall_torque / stall_current) to ensure physical consistency.

In [ ]:
# Motor configuration paths
from pathlib import Path

# Get absolute paths to XML files
notebook_dir = Path.cwd()
models_dir = notebook_dir / "notebooks" / "electrical" / "models"

# If we're already in the notebooks directory, adjust path
if not models_dir.exists():
  models_dir = notebook_dir / "electrical" / "models"

# If still not found, try from repo root
if not models_dir.exists():
  import os

  # Try to find repo root (has src/ directory)
  current = Path.cwd()
  while current != current.parent:
    if (current / "src" / "mjlab").exists():
      models_dir = current / "notebooks" / "electrical" / "models"
      break
    current = current.parent

motor_configs = {
  "underpowered": str(models_dir / "cartpole_underpowered.xml"),
  "well-sized": str(models_dir / "cartpole_good.xml"),
  "overpowered": str(models_dir / "cartpole_overpowered.xml"),
}

# Motor specifications for reference
motor_specs = {
  "underpowered": {"peak_torque": 20.0, "color": "red"},
  "well-sized": {"peak_torque": 60.0, "color": "green"},
  "overpowered": {"peak_torque": 150.0, "color": "blue"},
}

print("Motor configurations loaded:")
for name, spec in motor_specs.items():
  print(f"  {name:15s}: {spec['peak_torque']:5.0f} N⋅m peak")

print(f"\nModel directory: {models_dir}")
print(f"Models exist: {models_dir.exists()}")

## Step 2: Define Torque Command

We'll apply a sinusoidal torque command to exercise the motor through cyclic loading:

$$\tau(t) = A \sin(2\pi f t)$$

Where:
- **A = 15 N⋅m** (amplitude)
- **f = 1 Hz** (frequency, one full oscillation per second)

This oscillates the cart back and forth, forcing the motor to repeatedly accelerate and decelerate -- a demanding test of electrical characteristics.

In [ ]:
def sinusoidal_torque(t, amplitude=15.0, frequency=1.0):
  """Generate sinusoidal torque command for cyclic load testing.

  Args:
      t: Time in seconds
      amplitude: Torque amplitude in N⋅m
      frequency: Frequency in Hz

  Returns:
      Torque command in N⋅m
  """
  return amplitude * np.sin(2 * np.pi * frequency * t)


# Visualize the torque profile
t = np.linspace(0, 5, 500)
torque = sinusoidal_torque(t)

plt.figure(figsize=(10, 4))
plt.plot(t, torque, "k-", linewidth=2)
plt.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
plt.xlabel("Time (s)", fontsize=12)
plt.ylabel("Torque Command (N⋅m)", fontsize=12)
plt.title(
  "Sinusoidal Torque Command (Cyclic Load Test)", fontsize=14, fontweight="bold"
)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Torque command: 15 N⋅m amplitude at 1 Hz")

## Step 3: Simulation Function

This function:
1. Creates a CartPole environment with the specified motor
2. Applies sinusoidal torque commands
3. Records electrical metrics (current, power, temperature)
4. Returns time-series data for analysis

In [ ]:
def simulate_cartpole(xml_path, duration=5.0):
  """Run CartPole with constant rotation and record metrics.

  Args:
      xml_path: Path to CartPole XML with motor configuration
      duration: Simulation duration in seconds

  Returns:
      Dictionary of time-series data
  """
  # Create environment with custom motor
  cfg = cartpole_constant_rotation_env_cfg(xml_path, play=True)
  env = ManagerBasedRlEnv(cfg, device="cpu")

  # Storage for results
  results = {
    "time": [],
    "torque_command": [],
    "cart_position": [],
    "motor_current": [],
    "motor_power": [],
    "motor_temperature": [],
    "battery_voltage": [],
  }

  # Reset environment
  obs, info = env.reset()

  # Get metric indices
  metric_names = env.metrics_manager._term_names
  motor_current_idx = metric_names.index("motor_current_avg")
  motor_power_idx = metric_names.index("motor_power_total")
  motor_temp_idx = metric_names.index("motor_temperature_max")
  battery_voltage_idx = metric_names.index("battery_voltage")

  # Calculate timestep
  dt = cfg.sim.mujoco.timestep * cfg.decimation  # Control timestep
  steps = int(duration / dt)

  print(f"  Simulating {steps} steps ({duration:.1f}s)...")

  for step in range(steps):
    t = step * dt

    # Generate torque command
    torque = sinusoidal_torque(t)
    action = torch.tensor([[torque]])

    # Step environment (returns: obs, reward, terminated, truncated, info)
    obs, reward, terminated, truncated, info = env.step(action)

    # Record metrics from metrics_manager
    metric_values = env.metrics_manager._step_values[0]  # First (and only) environment

    results["time"].append(t)
    results["torque_command"].append(torque)
    results["cart_position"].append(obs["actor"][0, 0].item())  # cart_pos
    results["motor_current"].append(metric_values[motor_current_idx].item())
    results["motor_power"].append(metric_values[motor_power_idx].item())
    results["motor_temperature"].append(metric_values[motor_temp_idx].item())
    results["battery_voltage"].append(metric_values[battery_voltage_idx].item())

    # Progress indicator
    if step % 500 == 0 and step > 0:
      print(f"    Progress: {t:.1f}s / {duration:.1f}s")

    # Reset if episode ends
    if terminated or truncated:
      obs, info = env.reset()

  print(f"  ✓ Simulation complete")

  # Convert to numpy arrays
  return {k: np.array(v) for k, v in results.items()}


print("✓ Simulation function defined")

## Step 4: Run Simulations

Now we'll run the same task with all three motors and record their performance.

In [ ]:
# Run simulations for all three motors
results = {}

print("Running simulations...\n")

for name, xml_path in motor_configs.items():
  print(f"Motor: {name}")
  results[name] = simulate_cartpole(xml_path, duration=5.0)
  print()

print("✓ All simulations complete!")

## Step 5: Visualize Results

Let's compare the three motors across multiple metrics:

1. **Motor Current** - How much current each motor draws
2. **Power Consumption** - Total electrical power used
3. **Motor Temperature** - Thermal heating from I²R losses
4. **Average Power** - Overall efficiency comparison

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
fig.suptitle(
  "Motor Sizing Comparison: CartPole with Sinusoidal Torque",
  fontsize=16,
  fontweight="bold",
)

motor_names = ["underpowered", "well-sized", "overpowered"]

# Plot 1: Torque Command & Cart Position
ax = axes[0, 0]
for name in motor_names:
  color = motor_specs[name]["color"]
  ax.plot(
    results[name]["time"],
    results[name]["torque_command"],
    label=name.capitalize(),
    color=color,
    linewidth=2,
  )
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
ax.set_xlabel("Time (s)", fontsize=11)
ax.set_ylabel("Torque (N⋅m)", fontsize=11)
ax.set_title("Torque Command", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Cart Position (X axis)
ax = axes[0, 1]
for name in motor_names:
  color = motor_specs[name]["color"]
  ax.plot(
    results[name]["time"],
    results[name]["cart_position"],
    label=name.capitalize(),
    color=color,
    linewidth=2,
  )
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
ax.set_xlabel("Time (s)", fontsize=11)
ax.set_ylabel("Position (m)", fontsize=11)
ax.set_title("Cart Position (X axis)", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Motor Current
ax = axes[1, 0]
for name in motor_names:
  color = motor_specs[name]["color"]
  ax.plot(
    results[name]["time"],
    results[name]["motor_current"],
    label=name.capitalize(),
    color=color,
    linewidth=2,
  )
ax.set_xlabel("Time (s)", fontsize=11)
ax.set_ylabel("Current (A)", fontsize=11)
ax.set_title("Motor Current Draw", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 4: Power Consumption
ax = axes[1, 1]
for name in motor_names:
  color = motor_specs[name]["color"]
  ax.plot(
    results[name]["time"],
    results[name]["motor_power"],
    label=name.capitalize(),
    color=color,
    linewidth=2,
  )
ax.set_xlabel("Time (s)", fontsize=11)
ax.set_ylabel("Power (W)", fontsize=11)
ax.set_title("Power Consumption", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 5: Motor Temperature
ax = axes[2, 0]
for name in motor_names:
  color = motor_specs[name]["color"]
  ax.plot(
    results[name]["time"],
    results[name]["motor_temperature"],
    label=name.capitalize(),
    color=color,
    linewidth=2,
  )
ax.set_xlabel("Time (s)", fontsize=11)
ax.set_ylabel("Temperature (°C)", fontsize=11)
ax.set_title("Motor Temperature", fontsize=12, fontweight="bold")
ax.axhline(y=25, color="gray", linestyle="--", alpha=0.5, label="Ambient")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 6: Average Power Bar Chart
ax = axes[2, 1]
names = ["Underpowered", "Well-sized", "Overpowered"]
avg_powers = [
  np.mean(results["underpowered"]["motor_power"]),
  np.mean(results["well-sized"]["motor_power"]),
  np.mean(results["overpowered"]["motor_power"]),
]
colors = [
  motor_specs[name.lower()]["color"]
  for name in ["underpowered", "well-sized", "overpowered"]
]
bars = ax.bar(
  names, avg_powers, color=colors, alpha=0.7, edgecolor="black", linewidth=1.5
)
ax.set_ylabel("Average Power (W)", fontsize=11)
ax.set_title("Average Power Consumption", fontsize=12, fontweight="bold")
ax.grid(True, axis="y", alpha=0.3)

# Add value labels on bars
for bar, power in zip(bars, avg_powers):
  height = bar.get_height()
  ax.text(
    bar.get_x() + bar.get_width() / 2.0,
    height,
    f"{power:.1f}W",
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold",
  )

plt.tight_layout()
plt.show()

print("✓ Visualization complete!")

## Step 6: Metrics Summary

Let's compare the key metrics in a table:

In [ ]:
# Calculate metrics for all motors
print("\n" + "=" * 70)
print("MOTOR SIZING COMPARISON SUMMARY")
print("=" * 70)
print(
  f"\n{'Metric':<25s} {'Underpowered':>15s} {'Well-sized':>15s} {'Overpowered':>15s}"
)
print("-" * 70)

# Peak torque
print(f"{'Peak Torque (N⋅m)':<25s} {20.0:>15.0f} {60.0:>15.0f} {150.0:>15.0f}")

# Average current
avg_currents = {
  name: np.mean(results[name]["motor_current"])
  for name in ["underpowered", "well-sized", "overpowered"]
}
print(
  f"{'Avg Current (A)':<25s} {avg_currents['underpowered']:>15.2f} {avg_currents['well-sized']:>15.2f} {avg_currents['overpowered']:>15.2f}"
)

# Average power
avg_powers_dict = {
  name: np.mean(results[name]["motor_power"])
  for name in ["underpowered", "well-sized", "overpowered"]
}
print(
  f"{'Avg Power (W)':<25s} {avg_powers_dict['underpowered']:>15.1f} {avg_powers_dict['well-sized']:>15.1f} {avg_powers_dict['overpowered']:>15.1f}"
)

# Max temperature
max_temps = {
  name: np.max(results[name]["motor_temperature"])
  for name in ["underpowered", "well-sized", "overpowered"]
}
print(
  f"{'Max Temperature (°C)':<25s} {max_temps['underpowered']:>15.1f} {max_temps['well-sized']:>15.1f} {max_temps['overpowered']:>15.1f}"
)

# Load factor (commanded torque / peak torque)
load_factors = {
  "underpowered": 15.0 / 20.0 * 100,
  "well-sized": 15.0 / 60.0 * 100,
  "overpowered": 15.0 / 150.0 * 100,
}
print(
  f"{'Load Factor (%)':<25s} {load_factors['underpowered']:>15.0f} {load_factors['well-sized']:>15.0f} {load_factors['overpowered']:>15.0f}"
)

print("-" * 70)

# Normalized values (relative to well-sized)
print("\n" + "=" * 70)
print("NORMALIZED TO WELL-SIZED MOTOR (= 1.00)")
print("=" * 70)
print(
  f"\n{'Metric':<25s} {'Underpowered':>15s} {'Well-sized':>15s} {'Overpowered':>15s}"
)
print("-" * 70)

baseline_power = avg_powers_dict["well-sized"]
baseline_current = avg_currents["well-sized"]

print(
  f"{'Power Ratio':<25s} "
  f"{avg_powers_dict['underpowered'] / baseline_power:>15.2f} "
  f"{1.00:>15.2f} "
  f"{avg_powers_dict['overpowered'] / baseline_power:>15.2f}"
)
print(
  f"{'Current Ratio':<25s} "
  f"{avg_currents['underpowered'] / baseline_current:>15.2f} "
  f"{1.00:>15.2f} "
  f"{avg_currents['overpowered'] / baseline_current:>15.2f}"
)

print("-" * 70)
print()

## Step 7: Vizer Visualization (Colab Only)

**Interactive 3D visualization of the CartPole with well-sized motor**

This will:
- Launch a Viser server for real-time 3D rendering
- Embed the visualization in an iframe
- Allow you to interact with the simulation (play/pause, reset, camera controls)

In [ ]:
import subprocess
import re

# Note: Only works in Colab, skip in local environments
if IN_COLAB:
  print("🎬 Starting 3D visualization...")

  # Launch visualization server for well-sized motor
  process = subprocess.Popen(
    [
      "python",
      "-m",
      "mjlab.scripts.play",
      "notebooks/electrical/models/cartpole_good.xml",
      "--num_envs",
      "1",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
  )

  # Extract port from viser server output
  detected_port = None
  for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

    # Look for port in format ":8081"
    port_match = re.search(r":(\d{4})", line)
    if port_match and ("viser" in line.lower() or "serving" in line.lower()):
      detected_port = int(port_match.group(1))
      print(f"\n✅ Viser server running on port {detected_port}")
      break

  # Expose as iframe in Colab
  from google.colab import output

  port = detected_port if detected_port else 8081
  output.serve_kernel_port_as_iframe(port=port, height=700)

  print("\n🎮 Use the UI controls to:")
  print("   - Play/Pause simulation")
  print("   - Reset environment")
  print("   - Adjust playback speed")
  print("   - Rotate camera view")
else:
  print("⚠️ Vizer visualization only available in Google Colab")
  print("   Run locally with: python -m mjlab.scripts.play <config>")

## Key Insights

### Electrical Performance: Bigger Motor = Better Efficiency

The results are clear -- for a fixed torque demand, a larger motor is always more electrically efficient:

| Metric | Underpowered (20 N⋅m) | Well-sized (60 N⋅m) | Overpowered (150 N⋅m) |
|--------|----------------------|---------------------|----------------------|
| **Avg Power** | 39.2 W | 5.0 W | 2.2 W |
| **Peak Current** | ~12 A | ~6 A | ~4 A |
| **Temperature Rise** | +0.7 °C | +0.1 °C | +0.05 °C |
| **Cart Displacement** | Similar | Similar | Similar |

This makes physical sense: a larger motor has a higher Kt (more torque per amp), so it draws **less current** for the same torque. Less current means less I²R heating and less power wasted.

**So why not always use the biggest motor?**

### The Real Tradeoff: Weight, Cost, and Size

This simulation models the motor's *electrical* behavior, but it does **not** model the motor's *mass*. In a real robot, motor weight matters enormously:

| Motor | Peak Torque | Typical Mass | Typical Cost | Volume |
|-------|------------|-------------|-------------|--------|
| Underpowered | 20 N⋅m | ~0.3 kg | $ | Small |
| Well-sized | 60 N⋅m | ~1.2 kg | $$ | Medium |
| Overpowered | 150 N⋅m | ~4.0 kg | $$$$ | Large |

In a walking robot with 12+ joints, choosing a 150 N⋅m motor where 60 N⋅m suffices adds **~34 kg** of unnecessary mass. That extra mass:
- Requires **bigger batteries** (more weight, more cost)
- Increases the **torque demand on every other joint** (cascading effect)
- Reduces **agility and payload capacity**
- Increases **material and manufacturing cost**

### The 2-4× Sizing Rule (Revisited)

The rule isn't about minimizing electrical power -- it's about the **system-level tradeoff**:

> **Choose the smallest motor that comfortably handles your peak loads.**
>
> "Comfortably" means operating at 25-50% of peak torque (2-4× headroom).

- **Underpowered (75% load)**: Motor saturates at peaks, draws excessive current, overheats. Unreliable.
- **Well-sized (25% load)**: Handles peaks with margin. Reasonable weight and cost. Good thermal headroom.
- **Overpowered (10% load)**: Electrically perfect, but heavy and expensive. Wastes mechanical budget.

### Summary

| | Underpowered | Well-sized | Overpowered |
|---|---|---|---|
| **Electrical efficiency** | Poor | Good | Excellent |
| **Weight** | Light | Moderate | Heavy |
| **Cost** | Low | Moderate | High |
| **Reliability** | Poor (overheats) | Good | Excellent |
| **System-level optimum?** | No (saturates) | **Yes** | No (too heavy) |

The well-sized motor is the **system optimum**: it balances electrical performance against the weight and cost penalty. The overpowered motor is electrically superior but mechanically wasteful.

**Bottom line**: Motor sizing is not about minimizing power consumption -- it's about minimizing **total system weight and cost** while maintaining sufficient torque headroom.

## 🎓 Next Steps

1. **Try different torque profiles**: Modify the sinusoidal command (amplitude, frequency)
2. **Add motor mass to simulation**: Model the motor weight as additional cart mass to see the cascading effect
3. **Mission planning**: Predict battery runtime for different motor choices
4. **Train RL policies**: Use this setup with reinforcement learning

## 📚 Resources

- [mjlab Repository](https://github.com/robomotic/mjlab/tree/feature/motor-database-extension)
- [Tutorial 1: Motor Physics](01_intro.ipynb)
- [Design Proposal](https://github.com/robomotic/mjlab/blob/feature/motor-database-extension/docs/motors/design-proposal.md)

---

**Motor sizing = the art of picking the smallest motor you can get away with.**

Created with [mjlab](https://github.com/robomotic/mjlab)